In [1]:
import pandas as pd
import json

In [ ]:
with open('../data/processed/productos_con_nutricion.json', 'r', encoding='utf-8') as file :
  df = pd.read_json(file)

In [10]:
import os
from huggingface_hub import hf_hub_download

model_name = "TheBloke/TinyLlama-1.1B-Chat-v1.0-GGUF" # Changed repository
model_file = "tinyllama-1.1b-chat-v1.0.Q5_K_M.gguf" # Example GGUF file

# Define the path where the model will be downloaded
model_path = os.path.join(".", model_file)

# Check if the model already exists to avoid re-downloading
if not os.path.exists(model_path):
    print(f"Downloading {model_file} from {model_name}...")
    hf_hub_download(
        repo_id=model_name,
        filename=model_file,
        local_dir=".", # Download to the current directory
        local_dir_use_symlinks=False
    )
    print("Download complete.")
else:
    print(f"{model_file} already exists. Skipping download.")

C:\Users\dayan\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


C:\Users\dayan\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\huggingface_hub\file_download.py:986: UserWarning: `local_dir_use_symlinks` parameter is deprecated and will be ignored. The process to download files to a local folder has been updated and do not rely on symlinks anymore. You only need to pass a destination folder as`local_dir`.
For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/download#download-files-to-local-folder.
  warnings.warn(
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


Download complete.


### Preparación de Datos: Extracción de Información Nutricional

Primero, vamos a normalizar la columna informacion_nutricional extrayendo sus claves en nuevas columnas. Esto facilitará el filtrado de productos según necesidades dietéticas, como apto_vegetariano o azucares_g.

In [ ]:
# Expand the 'informacion_nutricional' dictionary into separate columns
nutritional_df = df['informacion_nutricional'].apply(pd.Series)

# Join the new nutritional columns back to the original DataFrame
df = pd.concat([df.drop('informacion_nutricional', axis=1), nutritional_df], axis=1)

# Drop the 'nota' column as it might not be directly useful for filtering
if 'nota' in df.columns:
    df = df.drop('nota', axis=1)

# Inspect the first few rows with the new columns and check data types
print("DataFrame after extracting nutritional information:")
display(df.head())
print("\nDataFrame Info:")
df.info()

In [ ]:
# Ensure 'precio_cop' is numeric
df['precio_cop'] = pd.to_numeric(df['precio_cop'], errors='coerce')

# Drop rows where price is missing after conversion (if any)
df.dropna(subset=['precio_cop'], inplace=True)

print("\nDataFrame with 'precio_cop' column:")
display(df[['nombre', 'precio_cop', 'calorias_kcal', 'apto_vegetariano']].head())

### Construcción del Agente Asistente de Compras

Esta función filtrará tu DataFrame de productos (df) según un presupuesto específico y preferencias dietéticas o nutricionales. Luego, formateará la información relevante de los productos en una cadena de texto y la enviará, junto con la solicitud del usuario, al modelo Llama para generar una sugerencia personalizada de cesta de compra.

In [ ]:
from llama_cpp import Llama

llm = Llama(model_path=model_path, n_ctx=2048, n_gpu_layers=-1)

In [ ]:
df[df['nombre'].str.contains('talco', case = False, na= False)]

In [ ]:
def shopping_assistant_agent(user_request: str, budget: float = None, preferences: dict = None, max_products_for_context: int = 10) -> str:
    """
    Genera una sugerencia de cesta de compra basada en la solicitud del usuario, su presupuesto y sus preferencias, utilizando el modelo Llama y los datos de productos.

    Parámetros:
        user_request (str): Consulta del usuario (por ejemplo, "Necesito productos veganos para una cena con un presupuesto de 50.000 COP").
        budget (float, opcional): Presupuesto máximo en COP. Por defecto es None.
        preferences (dict, opcional): Diccionario con preferencias dietéticas o nutricionales.
                                    Ejemplos: {'apto_vegetariano': True, 'azucares_g_max': 10}.
        max_products_for_context (int): Número máximo de productos que se incluirán en el contexto del prompt
                                        para evitar exceder los límites de tokens.

    Retorna:
        str: La sugerencia de cesta de compra generada por el agente.
    """
    filtered_df = df.copy()

    # Apply budget filter
    if budget is not None and 'precio_cop' in filtered_df.columns:
        filtered_df = filtered_df[filtered_df['precio_cop'] <= budget]

    # Apply dietary/nutritional preferences
    if preferences:
        for pref, value in preferences.items():
            if pref in filtered_df.columns:
                if isinstance(value, bool): # For boolean preferences like apto_vegetariano
                    filtered_df = filtered_df[filtered_df[pref] == value]
                elif '_max' in pref: # For max values like azucares_g_max
                    original_col = pref.replace('_max', '')
                    if original_col in filtered_df.columns:
                        filtered_df = filtered_df[filtered_df[original_col] <= value]
                elif '_min' in pref: # For min values like proteinas_g_min
                    original_col = pref.replace('_min', '')
                    if original_col in filtered_df.columns:
                        filtered_df = filtered_df[filtered_df[original_col] >= value]
                # Add more preference types as needed

    # Select relevant columns for the LLM and convert to string format
    # Limit the number of products passed to avoid exceeding context window
    if not filtered_df.empty:
        product_columns = ['nombre', 'marca', 'precio_cop', 'calorias_kcal', 'apto_vegetariano', 'azucares_g']
        available_columns = [col for col in product_columns if col in filtered_df.columns]

        # Take a sample if there are too many products
        if len(filtered_df) > max_products_for_context:
            context_products = filtered_df[available_columns].sample(max_products_for_context, random_state=42)
        else:
            context_products = filtered_df[available_columns]

        products_str = context_products.to_markdown(index=False)
        # Optionally, you can convert to a list of dictionaries for a different format
        # products_str = json.dumps(context_products.to_dict(orient='records'), indent=2)
    else:
        products_str = "No se encontraron productos que coincidan con los criterios."

    # Construct the prompt for the Llama model
    system_message = (
        "Eres un asistente de compras inteligente. Tu tarea es ayudar a los usuarios a crear una canasta "
        "de productos basándose en sus restricciones presupuestarias y preferencias dietéticas. "
        "Analiza la lista de productos proporcionada y sugiere una canasta equilibrada que satisfaga los requisitos del usuario. "
        "Si no hay productos que cumplan los criterios, indícalo claramente. Proporciona una lista concisa de los productos sugeridos."
    )

    user_prompt = f"""
    {user_request}

    Considera los siguientes productos disponibles:
    {products_str}

    Por favor, sugiere una canasta de compras basada en estos productos.
    """

    # Generate response using the Llama model
    output = llm.create_chat_completion(
        messages=[
            {"role": "system", "content": system_message},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.7,
        max_tokens=512 # Adjust max_tokens for potentially longer responses
    )
    return output["choices"][0]["message"]["content"]

# Example Usage:
print("\n--- Ejemplo 1: Productos vegetarianos con presupuesto --- ")
user_query_1 = "Necesito una canasta de productos vegetarianos con un presupuesto máximo de 20000 COP."
agent_response_1 = shopping_assistant_agent(
    user_query_1,
    budget=20000,
    preferences={'apto_vegetariano': True}
)
print(f"Usuario: {user_query_1}")
print(f"Asistente: {agent_response_1}")


print("\n--- Ejemplo 2: Productos bajos en azúcar --- ")
user_query_2 = "Quiero una canasta con productos bajos en azúcar, sin restricciones de presupuesto."
agent_response_2 = shopping_assistant_agent(
    user_query_2,
    preferences={'azucares_g_max': 5}
)
print(f"Usuario: {user_query_2}")
print(f"Asistente: {agent_response_2}")


print("\n--- Ejemplo 3: Productos sin coincidencias --- ")
user_query_3 = "Busco productos veganos con menos de 1g de grasa y un presupuesto de 100 COP."
agent_response_3 = shopping_assistant_agent(
    user_query_3,
    budget=100,
    preferences={'apto_vegano': True, 'grasas_g_max': 1}
)
print(f"Usuario: {user_query_3}")
print(f"Asistente: {agent_response_3}")


print("\n--- Ejemplo 4: Productos sin coincidencias --- ")
user_query_4 = "Busco productos saludables para hacer un desayuno con un presupuesto de 50000 COP. para dos personas"
agent_response_4 = shopping_assistant_agent(
    user_query_4,
    budget=50000,
    preferences={'grasas_g_max': 2000}
)
print(f"Usuario: {user_query_3}")
print(f"Asistente: {agent_response_3}")

### Carga del Modelo y Configuración del Agente

Ahora, carguemos el modelo utilizando Llama.from_pretrained y configuremos una interacción sencilla de tipo agente.

In [ ]:
from llama_cpp import Llama


def agent_response(prompt: str) -> str:
    """Generates a response from the Llama model based on the given prompt."""
    output = llm.create_chat_completion(
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.7,
        max_tokens=256 # Limit the response length
    )
    return output["choices"][0]["message"]["content"]

# Ejemplo de uso:
user_query = "Hello, what can you do for me?"
print(f"User: {user_query}")
agent_reply = agent_response(user_query)
print(f"Agent: {agent_reply}")

user_query_2 = "What is the capital of France?"
print(f"User: {user_query_2}")
agent_reply_2 = agent_response(user_query_2)
print(f"Agent: {agent_reply_2}")
